In [ ]:
#functions in building oracle
def controlled_add_const_in_qft(qc: QuantumCircuit, ctrl, acc, const: int):
    """
    Adds classical constant `const` to `acc` IN THE QFT BASIS, controlled on `ctrl`.
    Assumes acc is little-endian (acc[0]=LSB).
    """
    n = len(acc)
    const %= (1 << n)
    if const == 0:
        return

    for k in range(n):
        angle = 0.0
        for j in range(k + 1):
            if (const >> j) & 1:
                angle += 2 * math.pi / (1 << (k - j + 1))
        if angle != 0.0:
            qc.cp(angle, ctrl, acc[k])


def add_subset_sum_into_acc(qc: QuantumCircuit, subset, acc, xs):
    """
    Compute sum_i subset[i]*xs[i] into acc (acc starts |0>), then return acc to comp basis.
    """
    qft_no_swaps(qc, acc)
    for i, xi in enumerate(xs):
        controlled_add_const_in_qft(qc, subset[i], acc, xi)
    iqft_no_swaps(qc, acc)


def add_const(qc: QuantumCircuit, acc, const: int):
    """
    Uncontrolled add constant to acc (computational basis) using QFT phases.
    """
    m = len(acc)
    const %= (1 << m)
    if const == 0:
        return

    qft_no_swaps(qc, acc)
    for k in range(m):
        angle = 0.0
        for j in range(k + 1):
            if (const >> j) & 1:
                angle += 2 * math.pi / (1 << (k - j + 1))
        if angle != 0.0:
            qc.p(angle, acc[k])
    iqft_no_swaps(qc, acc)


def sub_const(qc: QuantumCircuit, acc, const: int):
    """
    Subtract constant from acc (computational basis).
    """
    m = len(acc)
    neg = ((1 << m) - (const % (1 << m))) % (1 << m)
    add_const(qc, acc, neg)

In [ ]:
#oracle construction

def build_subset_sum_phase_oracle(xs, t, m):
    n = len(xs)

    subset = QuantumRegister(n, "s")
    acc = QuantumRegister(m, "a")
    anc = AncillaRegister(max(1, m - 1), "anc")  
    ph = QuantumRegister(1, "ph")

    qc = QuantumCircuit(subset, acc, anc, ph, name="Oracle")
 
    qc.x(ph[0])
    qc.h(ph[0])

    # compute sum into acc
    add_subset_sum_into_acc(qc, subset, acc, xs)

    # acc := acc - t  (sum==t <=> acc==0)
    sub_const(qc, acc, t)

    # mark acc==0 by flipping acc bits then MCX onto phase(since MCX only works when all digits are 1)
    for q in acc:
        qc.x(q)
    mcx_generic(qc, list(acc), ph[0])
    for q in acc:
        qc.x(q)

    # uncompute: add t back
    add_const(qc, acc, t)

    # uncompute subset sum: add (-xi) controlled on subset[i]
    qft_no_swaps(qc, acc)
    for i, xi in enumerate(xs):
        neg_xi = ((1 << m) - (xi % (1 << m))) % (1 << m)
        controlled_add_const_in_qft(qc, subset[i], acc, neg_xi)
    iqft_no_swaps(qc, acc)

    # return phase qubit to |0> (optional)
    qc.h(ph[0])
    qc.x(ph[0])

    return qc